# 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Loading and Preprocessing


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

# 3. Build the Pretrained Model


In [ ]:
model = torchvision.models.vgg16(pretrained=True)

for param in model.features.parameters():
    param.requires_grad = False

num_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 10)
)

model = model.to(device)

# 4. Feature Extraction Training


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

num_epochs = 3
print_every = 100

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for step, (images, labels) in enumerate(trainloader, start=1):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if step % print_every == 0 or step == len(trainloader):
            avg_loss = running_loss / step
            print(f"Epoch [{epoch+1}/{num_epochs}] Step [{step}/{len(trainloader)}] Avg Loss: {avg_loss:.4f}", flush=True)

    epoch_loss = running_loss / len(trainloader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Completed - Loss: {epoch_loss:.4f}", flush=True)

Epoch [1/3] Step [100/1563] Avg Loss: 1.0778
Epoch [1/3] Step [200/1563] Avg Loss: 0.9103
Epoch [1/3] Step [300/1563] Avg Loss: 0.8455
Epoch [1/3] Step [400/1563] Avg Loss: 0.8005
Epoch [1/3] Step [500/1563] Avg Loss: 0.7629
Epoch [1/3] Step [600/1563] Avg Loss: 0.7367
Epoch [1/3] Step [700/1563] Avg Loss: 0.7217
Epoch [1/3] Step [800/1563] Avg Loss: 0.7086
Epoch [1/3] Step [900/1563] Avg Loss: 0.6966
Epoch [1/3] Step [1000/1563] Avg Loss: 0.6824
Epoch [1/3] Step [1100/1563] Avg Loss: 0.6726
Epoch [1/3] Step [1200/1563] Avg Loss: 0.6639
Epoch [1/3] Step [1300/1563] Avg Loss: 0.6546
Epoch [1/3] Step [1400/1563] Avg Loss: 0.6471
Epoch [1/3] Step [1500/1563] Avg Loss: 0.6417
Epoch [1/3] Step [1563/1563] Avg Loss: 0.6388
Epoch [1/3] Completed - Loss: 0.6388
Epoch [2/3] Step [100/1563] Avg Loss: 0.3913
Epoch [2/3] Step [200/1563] Avg Loss: 0.3992
Epoch [2/3] Step [300/1563] Avg Loss: 0.4008
Epoch [2/3] Step [400/1563] Avg Loss: 0.4035
Epoch [2/3] Step [500/1563] Avg Loss: 0.4122
Epoch [2/3]

# 5. Fine-Tuning


In [ ]:
for param in model.features[24:].parameters():
    param.requires_grad = True

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

num_epochs_ft = 3
print_every_ft = 100

for epoch in range(num_epochs_ft):
    model.train()
    running_loss = 0.0

    for step, (images, labels) in enumerate(trainloader, start=1):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if step % print_every_ft == 0 or step == len(trainloader):
            avg_loss = running_loss / step
            print(f"Fine-tune Epoch [{epoch+1}/{num_epochs_ft}] Step [{step}/{len(trainloader)}] Avg Loss: {avg_loss:.4f}", flush=True)

    epoch_loss = running_loss / len(trainloader)
    print(f"Fine-tune Epoch [{epoch+1}/{num_epochs_ft}] Completed - Loss: {epoch_loss:.4f}", flush=True)

Fine-tune Epoch [1/3] Step [100/1563] Avg Loss: 0.2392
Fine-tune Epoch [1/3] Step [200/1563] Avg Loss: 0.2412
Fine-tune Epoch [1/3] Step [300/1563] Avg Loss: 0.2334
Fine-tune Epoch [1/3] Step [400/1563] Avg Loss: 0.2293
Fine-tune Epoch [1/3] Step [500/1563] Avg Loss: 0.2245
Fine-tune Epoch [1/3] Step [600/1563] Avg Loss: 0.2229
Fine-tune Epoch [1/3] Step [700/1563] Avg Loss: 0.2207
Fine-tune Epoch [1/3] Step [800/1563] Avg Loss: 0.2227
Fine-tune Epoch [1/3] Step [900/1563] Avg Loss: 0.2225
Fine-tune Epoch [1/3] Step [1000/1563] Avg Loss: 0.2190
Fine-tune Epoch [1/3] Step [1100/1563] Avg Loss: 0.2172
Fine-tune Epoch [1/3] Step [1200/1563] Avg Loss: 0.2172
Fine-tune Epoch [1/3] Step [1300/1563] Avg Loss: 0.2158
Fine-tune Epoch [1/3] Step [1400/1563] Avg Loss: 0.2148
Fine-tune Epoch [1/3] Step [1500/1563] Avg Loss: 0.2135
Fine-tune Epoch [1/3] Step [1563/1563] Avg Loss: 0.2137
Fine-tune Epoch [1/3] Completed - Loss: 0.2137
Fine-tune Epoch [2/3] Step [100/1563] Avg Loss: 0.1700
Fine-tune E